In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np


def _find_content_slice(volume: np.ndarray, axis: int, fraction: float) -> int:
    """Return the slice index at fraction along axis, falling back to the
    nearest non-empty slice if the requested one is all zeros."""
    max_index = volume.shape[axis] - 1
    requested = int(round(fraction * max_index))

    if np.take(volume, indices=requested, axis=axis).any():
        return requested

    for delta in range(1, max_index + 1):
        for candidate in (requested - delta, requested + delta):
            if 0 <= candidate <= max_index:
                if np.take(volume, indices=candidate, axis=axis).any():
                    return candidate

    return requested  # volume is entirely zero


def _display_limits(volume: np.ndarray):
    """Return (vmin, vmax) over the non-zero voxels only.

    Uses 1st–99th percentile of non-zero voxels so the full tissue range is
    visible.  Background (0) is handled separately via masked arrays, so it
    always renders black regardless of whether vmin is negative.
    """
    nonzero = volume[volume != 0]
    if nonzero.size == 0:
        return 0.0, 1.0
    return float(np.percentile(nonzero, 1)), float(np.percentile(nonzero, 99))


def show_nth_slice(
    nii_path: str,
    slice_fraction: float,
    axis: int = 2,
    cmap: str = "gray",
    nii_path_2: str | None = None,
) -> None:
    """Display a slice from one or two 3D NIfTI volumes using a fractional position.

    Background (zero voxels) always renders black via masked arrays - works for
    both raw data (tissue > 0) and z-score normalised data (tissue spans negatives).
    Falls back to nearest non-empty slice if the requested slice is blank.

    Args:
        nii_path: Path to a .nii or .nii.gz file.
        slice_fraction: Fraction along the chosen axis in [0, 1].
        axis: Axis along which to slice (0, 1, or 2).
        cmap: Matplotlib colormap.
        nii_path_2: Optional second path to display side by side.
    """

    def _extract(path_str: str):
        file_path = Path(path_str)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        img    = nib.load(str(file_path))
        volume = img.get_fdata()
        if volume.ndim < 3:
            raise ValueError(f"Expected ≥3D data, got {volume.shape}")
        idx  = _find_content_slice(volume, axis, slice_fraction)
        slc  = np.take(volume, indices=idx, axis=axis)
        # Mask zeros so background renders black regardless of vmin/vmax
        slc_masked = np.ma.masked_where(slc == 0, slc)
        vmin, vmax = _display_limits(volume)
        return file_path.name, slc_masked, volume.shape, idx, vmin, vmax

    def _make_cmap(name: str):
        cm = plt.cm.get_cmap(name).copy()
        cm.set_bad("black")  # masked (background) pixels become black
        return cm

    if axis not in (0, 1, 2):
        raise ValueError("axis must be 0, 1, or 2")
    if not (0.0 <= slice_fraction <= 1.0):
        raise ValueError("slice_fraction must be in [0, 1]")

    name_1, slc_1, shape_1, idx_1, vmin_1, vmax_1 = _extract(nii_path)
    print(f"{name_1}  shape={shape_1}  frac={slice_fraction:.3f} → idx={idx_1}  "
          f"display=[{vmin_1:.2f}, {vmax_1:.2f}]")

    cm = _make_cmap(cmap)

    if nii_path_2 is None:
        plt.figure(figsize=(6, 6))
        plt.imshow(slc_1.T, cmap=cm, origin="lower", vmin=vmin_1, vmax=vmax_1)
        plt.title(f"{name_1} | axis={axis} | frac={slice_fraction:.3f} | idx={idx_1}")
        plt.axis("off")
        plt.show()
        return

    name_2, slc_2, shape_2, idx_2, vmin_2, vmax_2 = _extract(nii_path_2)
    print(f"{name_2}  shape={shape_2}  frac={slice_fraction:.3f} → idx={idx_2}  "
          f"display=[{vmin_2:.2f}, {vmax_2:.2f}]")

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(slc_1.T, cmap=cm, origin="lower", vmin=vmin_1, vmax=vmax_1)
    axes[0].set_title(f"{name_1} | frac={slice_fraction:.3f} | idx={idx_1}")
    axes[0].axis("off")
    axes[1].imshow(slc_2.T, cmap=cm, origin="lower", vmin=vmin_2, vmax=vmax_2)
    axes[1].set_title(f"{name_2} | frac={slice_fraction:.3f} | idx={idx_2}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
show_nth_slice(NII_1, slice_fraction=0.5, axis=2, nii_path_2=NII_2)